# Telegram Topic Ops Bot - Heroku Deploy (Colab)

Deploys your single control bot that supports both:

- `/export` (topic/channel export)
- `/clone` (topic-to-topic cloning)

This notebook now deploys from the `heroku_bot` folder in your GitHub repo, so only Heroku app files are pushed to Heroku.

Expected repository layout:

- `heroku_bot/app.py`
- `heroku_bot/Procfile`
- `heroku_bot/requirements.txt`
- `heroku_bot/runtime.txt`

---

## What You Need Before Running

- Heroku account + API key
- A Telegram bot token (`HEROKU_BOT_TOKEN`)
- Telegram user session string (`TG_SESSION_STRING`)
- MongoDB URI (`MONGODB_URI`) or MongoDB Data API credentials
- Your Telegram numeric user id for admin access (`BOT_ADMIN_USER_IDS`)

In [ ]:
#@title 1) Install Heroku CLI + Login { display-mode: "form" }

Heroku_Email = "" #@param {type:"string"}
Heroku_API_Key = "" #@param {type:"string"}

if not Heroku_Email or not Heroku_API_Key:
    raise ValueError("Please provide Heroku_Email and Heroku_API_Key")

!curl -s https://cli-assets.heroku.com/install.sh | sh

from os import path as ospath, chmod

netrc_path = ospath.expanduser("~/.netrc")
netrc_creds = f'''machine api.heroku.com
  login {Heroku_Email}
  password {Heroku_API_Key}
machine git.heroku.com
  login {Heroku_Email}
  password {Heroku_API_Key}'''

with open(netrc_path, "w", encoding="utf-8") as f:
    f.write(netrc_creds)

chmod(netrc_path, 0o600)

!git config --global user.email "{Heroku_Email}"
!git config --global user.name "Telegram Topic Ops Bot"

print("Heroku CLI installed and login credentials configured.")
!heroku auth:whoami

In [ ]:
#@title 2) Create Heroku App { display-mode: "form" }

App_Name = "" #@param {type:"string"}
Server_Region = "eu" #@param ["eu", "us"]
Heroku_Team = "" #@param {type:"string"}

team_arg = f"--team {Heroku_Team}" if Heroku_Team.strip() else ""
name_arg = App_Name.strip()

cmd = f"heroku create --stack heroku-24 --region {Server_Region} {team_arg} {name_arg}".strip()
print("Running:", cmd)
!{cmd}

print("App creation step finished.")

In [ ]:
#@title 3) Clone Repo + Build Deploy Workspace + Set Heroku Config Vars { display-mode: "form" }

# Heroku target app
App_Name = "" #@param {type:"string"}

# Repo to deploy
Repo_URL = "https://github.com/<your-username>/<your-repo>.git" #@param {type:"string"}
Repo_Branch = "main" #@param {type:"string"}

# Required bot/runtime vars
TG_API_ID = 0 #@param {type:"integer"}
TG_API_HASH = "" #@param {type:"string"}
TG_SESSION_STRING = "" #@param {type:"string"}
HEROKU_BOT_TOKEN = "" #@param {type:"string"}
BOT_ADMIN_USER_IDS = "" #@param {type:"string"}

# MongoDB (URI mode - recommended for your setup)
MONGODB_URI = "" #@param {type:"string"}
MONGODB_DATABASE = "topic_ops" #@param {type:"string"}
MONGODB_COLLECTION = "bot_state" #@param {type:"string"}

# Optional MongoDB Data API vars (leave blank when using MONGODB_URI)
MONGODB_DATA_API_URL = "" #@param {type:"string"}
MONGODB_DATA_API_KEY = "" #@param {type:"string"}
MONGODB_DATA_SOURCE = "" #@param {type:"string"}

if not App_Name.strip():
    raise ValueError("App_Name is required")

required = [TG_API_ID, TG_API_HASH, TG_SESSION_STRING, HEROKU_BOT_TOKEN, BOT_ADMIN_USER_IDS]
if any(v in ["", 0, None] for v in required):
    raise ValueError("Please fill all required Telegram/Heroku variables")

if not MONGODB_URI.strip() and not all([MONGODB_DATA_API_URL, MONGODB_DATA_API_KEY, MONGODB_DATA_SOURCE]):
    raise ValueError("Provide MONGODB_URI or complete MongoDB Data API variables")

import os
import shutil
import subprocess
from pathlib import Path

repo_dir = Path("/content/topic_ops_repo")
deploy_dir = Path("/content/topic_ops_heroku_deploy")

for path in [repo_dir, deploy_dir]:
    if path.exists():
        shutil.rmtree(path)

subprocess.check_call(["git", "clone", "-b", Repo_Branch, Repo_URL, str(repo_dir)])
print("Cloned repo at:", repo_dir)

bundle_dir = repo_dir / "heroku_bot"
if not bundle_dir.exists():
    raise FileNotFoundError("heroku_bot folder not found in repository")

deploy_dir.mkdir(parents=True, exist_ok=True)

# Copy only Heroku deploy files from heroku_bot into deploy root
for item in bundle_dir.iterdir():
    target = deploy_dir / item.name
    if item.name == "runtime" and item.is_dir():
        continue
    if item.is_dir():
        shutil.copytree(item, target, dirs_exist_ok=True)
    else:
        shutil.copy2(item, target)

# Ensure expected root-level files for Heroku
procfile = deploy_dir / "Procfile"
if not procfile.exists():
    procfile.write_text("worker: python app.py\n", encoding="utf-8")

runtime_file = deploy_dir / "runtime.txt"
if not runtime_file.exists():
    runtime_file.write_text("python-3.12.0\n", encoding="utf-8")

os.chdir(deploy_dir)
print("Prepared deploy workspace at:", deploy_dir)

config_pairs = {
    "TG_API_ID": str(TG_API_ID),
    "TG_API_HASH": TG_API_HASH,
    "TG_SESSION_STRING": TG_SESSION_STRING,
    "HEROKU_BOT_TOKEN": HEROKU_BOT_TOKEN,
    "BOT_ADMIN_USER_IDS": BOT_ADMIN_USER_IDS,
    "MONGODB_URI": MONGODB_URI,
    "MONGODB_DATABASE": MONGODB_DATABASE,
    "MONGODB_COLLECTION": MONGODB_COLLECTION,
}

# Add Data API keys only if user filled them
if MONGODB_DATA_API_URL.strip() and MONGODB_DATA_API_KEY.strip() and MONGODB_DATA_SOURCE.strip():
    config_pairs["MONGODB_DATA_API_URL"] = MONGODB_DATA_API_URL
    config_pairs["MONGODB_DATA_API_KEY"] = MONGODB_DATA_API_KEY
    config_pairs["MONGODB_DATA_SOURCE"] = MONGODB_DATA_SOURCE

cmd = ["heroku", "config:set", "-a", App_Name] + [f"{k}={v}" for k, v in config_pairs.items()]
subprocess.check_call(cmd)
print("Heroku config vars set successfully.")

In [ ]:
#@title 4) Deploy to Heroku { display-mode: "form" }

App_Name = "" #@param {type:"string"}

if not App_Name.strip():
    raise ValueError("App_Name is required")

import os
import subprocess
from pathlib import Path

deploy_path = Path("/content/topic_ops_heroku_deploy")
if not deploy_path.exists():
    raise FileNotFoundError("Deploy workspace not found. Run cell 3 first.")

os.chdir(deploy_path)
subprocess.check_call(["git", "init"])
subprocess.check_call(["git", "branch", "-M", "main"])
subprocess.check_call(["git", "add", ".", "-f"])
subprocess.run(["git", "commit", "-m", "Heroku deploy bundle"], check=False)
subprocess.check_call(["heroku", "git:remote", "-a", App_Name])
subprocess.check_call(["git", "push", "heroku", "main", "-f"])
subprocess.check_call(["heroku", "ps:scale", "worker=1", "-a", App_Name])

print("Deploy complete from heroku_bot bundle. Worker scaled to 1.")

In [ ]:
#@title 5) Show Heroku Logs { display-mode: "form" }

App_Name = "" #@param {type:"string"}

if not App_Name.strip():
    raise ValueError("App_Name is required")

!heroku logs --tail -a {App_Name}

In [ ]:
#@title 6) Heroku Logout

!heroku logout